# Paso 2 — EFA por bloques temáticos

**Input:** `dataset_filtrado.csv` (592 columnas, 6,968 estudiantes)  
**Output:** `dataset_efa_items_seleccionados.csv` — ítems con carga factorial ≥ 0.40  

**Flujo:**
1. Detectar prefijos y proponer bloques automáticamente
2. Confirmar/ajustar bloques (celda editable)
3. Imputar missing por bloque
4. Aplicar EFA por bloque
5. Seleccionar ítems con carga ≥ 0.40
6. Guardar dataset final para IRT y LCA

## 0. Librerías

Necesitas instalar `factor_analyzer` si no lo tienes:
```
pip install factor_analyzer
```

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity

# Umbral de carga factorial para retener un ítem
UMBRAL_CARGA = 0.40

print("Librerías cargadas.")

Librerías cargadas.


## 1. Cargar dataset filtrado

In [5]:
df = pd.read_csv("/workspace/notebooks/01_data_cleansing/dataset_filtrado.csv", low_memory=False)
print(f"Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]:,} columnas")

Dataset cargado: 6,968 filas × 592 columnas


## 2. Detectar prefijos automáticamente

PISA nombra sus variables con prefijos de 2–4 caracteres que indican el cuestionario de origen.  
Esta celda los detecta y muestra el conteo para que puedas decidir cómo agrupar.

In [ ]:
def extraer_prefijo(nombre, largo=2):
    """Extrae los primeros `largo` caracteres alfabéticos del nombre de columna."""
    letras = ''.join([c for c in nombre if c.isalpha()])
    return letras[:largo].upper() if len(letras) >= largo else nombre.upper()[:largo]

# Probar con prefijos de 2 y 3 caracteres
for largo in [2, 3]:
    prefijos = pd.Series([extraer_prefijo(c, largo) for c in df.columns])
    conteo = prefijos.value_counts()
    print(f"\nPrefijos de {largo} caracteres ({len(conteo)} grupos):")
    print(conteo.to_string())


Prefijos de 2 caracteres (32 grupos):
SC    291
ST    162
FL     93
RS      4
MS      4
RC      3
MA      3
EF      2
PR      2
CO      2
SR      2
MC      2
VE      2
MI      2
GR      1
PA      1
RD      1
MP      1
IS      1
LA      1
RE      1
SK      1
TA      1
EX      1
DU      1
SI      1
FI      1
HI      1
UN      1
CL      1
CR      1
AB      1

Prefijos de 3 caracteres (40 grupos):
SCQ    287
STQ    155
FLQ     93
STD      7
RCO      3
EFF      2
COB      2
SRE      2
RSL      2
RST      2
MSI      2
MSL      2
VER      2
MIS      2
MAT      2
SCH      2
SCS      2
PRO      1
GRA      1
PAR      1
RDE      1
MPA      1
MCO      1
ISC      1
LAN      1
REP      1
SKI      1
TAR      1
EXP      1
DUR      1
SIS      1
FIS      1
HIS      1
UNI      1
PRI      1
MCL      1
CLS      1
CRE      1
MAC      1
ABG      1


## 3. ── EDITA AQUÍ ── Definir bloques temáticos

Basándote en los prefijos detectados arriba, define tus bloques.  
Cada bloque es una lista de prefijos que pertenecen al mismo cuestionario.

**Referencia de prefijos PISA estándar:**
- `ST` → Cuestionario del estudiante (Student)
- `SC` → Cuestionario de la escuela (School)
- `TC` → Cuestionario del docente (Teacher)
- `IC` → Cuestionario TIC / tecnología
- `PA` → Cuestionario de padres (Parents)
- `EC` → Cuestionario educativo complementario

Ajusta los nombres y prefijos según lo que viste en la celda anterior.

In [ ]:
# ── AJUSTA ESTE DICCIONARIO según los prefijos que viste arriba ───────────────
#
# Formato:  "nombre_bloque" : ["PREFIJO1", "PREFIJO2", ...]
# Si un prefijo no encaja en ningún grupo conocido, ponlo en "otros"

# BLOQUES_PREFIJOS = {
#     "estudiante" : ["ST"],
#     "escuela"    : ["SC"],
#     "docente"    : ["TC"],
#     "tic"        : ["IC"],
#     "padres"     : ["PA"],
#     "otros"      : [],      # ← agrega aquí prefijos que no reconozcas
# }
BLOQUES_PREFIJOS = {
    "estudiante" : ["ST", "STQ", "STD"],

    "escuela"    : ["SC", "SCQ", "SCH", "SCS"],

    "docente"    : [],   # no aparece en tu dataset

    "tic"        : ["IC", "ISC"],

    "padres"     : ["PA", "PAR"],

    "otros"      : [
        "FL", "FLQ",
        "RS", "RSL", "RST",
        "MS", "MSI", "MSL",
        "RC", "RCO",
        "MA", "MAT", "MAC",
        "EF", "EFF",
        "PR", "PRO", "PRI",
        "CO", "COB",
        "SR", "SRE",
        "MC", "MCO", "MCL",
        "VE", "VER",
        "MI", "MIS",
        "GR", "GRA",
        "RD", "RDE",
        "MP", "MPA",
        "IS",
        "LA", "LAN",
        "RE", "REP",
        "SK", "SKI",
        "TA", "TAR",
        "EX", "EXP",
        "DU", "DUR",
        "SI", "SIS",
        "FI", "FIS",
        "HI", "HIS",
        "UN", "UNI",
        "CL", "CLS",
        "CR", "CRE",
        "AB", "ABG"
    ]
}
# ─────────────────────────────────────────────────────────────────────────────

# Asignar cada columna a su bloque
def asignar_bloque(nombre, bloques_dict, largo=2):
    pref = extraer_prefijo(nombre, largo)
    for bloque, prefijos in bloques_dict.items():
        if pref in [p.upper() for p in prefijos]:
            return bloque
    return "sin_clasificar"

bloque_por_col = {col: asignar_bloque(col, BLOQUES_PREFIJOS) for col in df.columns}
serie_bloques  = pd.Series(bloque_por_col)

print("Columnas por bloque:")
print(serie_bloques.value_counts().to_string())
print()
print("Nota: si 'sin_clasificar' tiene muchas columnas, revisa y agrega sus prefijos al diccionario.")

Columnas por bloque:
escuela       291
estudiante    162
otros         138
padres          1

Nota: si 'sin_clasificar' tiene muchas columnas, revisa y agrega sus prefijos al diccionario.


## 4. Función de EFA para un bloque

In [8]:
def efa_bloque(df_bloque, nombre_bloque, umbral_carga=0.40, max_factores=10):
    """
    Aplica EFA a un bloque de columnas.
    Retorna:
        - items_seleccionados : lista de columnas con carga >= umbral en al menos un factor
        - df_cargas           : DataFrame de cargas factoriales
        - n_factores          : número de factores seleccionados por scree
    """
    print(f"\n{'═'*55}")
    print(f"  BLOQUE: {nombre_bloque.upper()} ({df_bloque.shape[1]} columnas)")
    print(f"{'═'*55}")

    # Imputar missing con la mediana de cada columna
    df_imp = df_bloque.apply(lambda col: col.fillna(col.median()))

    # Necesitamos al menos 5 columnas para EFA
    if df_imp.shape[1] < 5:
        print("  ✗ Bloque con menos de 5 columnas — omitido.")
        return [], pd.DataFrame(), 0

    # ── Prueba de adecuación muestral ─────────────────────────────────────────
    try:
        chi2, p_bartlett = calculate_bartlett_sphericity(df_imp)
        kmo_all, kmo_model = calculate_kmo(df_imp)
        print(f"  KMO          : {kmo_model:.3f}  "
              f"({'adecuado' if kmo_model >= 0.6 else 'bajo — interpretar con cautela'})")
        print(f"  Bartlett p   : {p_bartlett:.4f}  "
              f"({'significativo ✓' if p_bartlett < 0.05 else 'no significativo ✗'})")
        if kmo_model < 0.5:
            print("  ✗ KMO < 0.5 — bloque no apto para EFA. Omitido.")
            return [], pd.DataFrame(), 0
    except Exception as e:
        print(f"  ✗ Error en pruebas de adecuación: {e}. Omitido.")
        return [], pd.DataFrame(), 0

    # ── Scree plot para determinar número de factores ─────────────────────────
    try:
        fa_scree = FactorAnalyzer(n_factors=min(df_imp.shape[1]-1, max_factores),
                                   rotation=None)
        fa_scree.fit(df_imp)
        eigenvalores, _ = fa_scree.get_eigenvalues()
        n_factores = max(1, int((eigenvalores > 1).sum()))
        n_factores = min(n_factores, max_factores, df_imp.shape[1] - 1)
        print(f"  Factores seleccionados (eigenvalor > 1): {n_factores}")
    except Exception as e:
        print(f"  ✗ Error en scree: {e}. Omitido.")
        return [], pd.DataFrame(), 0

    # ── Scree plot visual ─────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(range(1, len(eigenvalores)+1), eigenvalores, "o-", color="#4A90D9", markersize=5)
    ax.axhline(1, color="#E24B4A", linewidth=1, linestyle="--", label="Eigenvalor = 1")
    ax.axvline(n_factores, color="#27AE60", linewidth=1, linestyle=":",
               label=f"n_factores = {n_factores}")
    ax.set_title(f"Scree plot — bloque {nombre_bloque}")
    ax.set_xlabel("Factor")
    ax.set_ylabel("Eigenvalor")
    ax.legend(fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()

    # ── EFA con rotación oblicua (oblimin) ────────────────────────────────────
    # Oblimin es preferible a varimax cuando se espera correlación entre factores
    try:
        fa = FactorAnalyzer(n_factors=n_factores, rotation="oblimin")
        fa.fit(df_imp)
    except Exception as e:
        print(f"  ✗ Error en EFA: {e}. Intentando sin rotación...")
        try:
            fa = FactorAnalyzer(n_factors=n_factores, rotation=None)
            fa.fit(df_imp)
        except Exception as e2:
            print(f"  ✗ Error sin rotación también: {e2}. Omitido.")
            return [], pd.DataFrame(), 0

    # ── Cargas factoriales ────────────────────────────────────────────────────
    cargas = fa.loadings_
    cols_factores = [f"F{i+1}_{nombre_bloque}" for i in range(n_factores)]
    df_cargas = pd.DataFrame(cargas, index=df_bloque.columns, columns=cols_factores)

    # Seleccionar ítems con carga >= umbral en al menos un factor
    carga_max = df_cargas.abs().max(axis=1)
    items_seleccionados = carga_max[carga_max >= umbral_carga].index.tolist()

    print(f"  Ítems con carga ≥ {umbral_carga}: {len(items_seleccionados)} / {df_bloque.shape[1]}")

    # Heatmap de cargas
    if len(items_seleccionados) > 0:
        altura = max(3, len(items_seleccionados) * 0.3)
        fig, ax = plt.subplots(figsize=(max(4, n_factores*1.5), altura))
        sns.heatmap(
            df_cargas.loc[items_seleccionados].round(2),
            annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1,
            linewidths=0.3, ax=ax, annot_kws={"size": 8}
        )
        ax.set_title(f"Cargas factoriales — bloque {nombre_bloque} (ítems seleccionados)")
        plt.tight_layout()
        plt.show()

    return items_seleccionados, df_cargas, n_factores

## 5. Ejecutar EFA en todos los bloques

In [9]:
resultados_efa = {}   # bloque → {items, cargas, n_factores}
todos_items_seleccionados = []

for nombre_bloque, _ in BLOQUES_PREFIJOS.items():
    # Obtener columnas de este bloque
    cols_bloque = [c for c, b in bloque_por_col.items() if b == nombre_bloque]

    if len(cols_bloque) == 0:
        continue

    df_bloque = df[cols_bloque].copy()

    items, cargas, n_fact = efa_bloque(
        df_bloque,
        nombre_bloque=nombre_bloque,
        umbral_carga=UMBRAL_CARGA
    )

    resultados_efa[nombre_bloque] = {
        "items_seleccionados" : items,
        "cargas"              : cargas,
        "n_factores"          : n_fact,
        "n_columnas_bloque"   : len(cols_bloque),
    }
    todos_items_seleccionados.extend(items)

# Bloque sin_clasificar (si existe)
cols_sin_clasificar = [c for c, b in bloque_por_col.items() if b == "sin_clasificar"]
if cols_sin_clasificar:
    print(f"\n⚠ {len(cols_sin_clasificar)} columnas sin clasificar — revisa BLOQUES_PREFIJOS")
    print("Ejemplos:", cols_sin_clasificar[:10])


═══════════════════════════════════════════════════════
  BLOQUE: ESTUDIANTE (162 columnas)
═══════════════════════════════════════════════════════
  KMO          : 0.809  (adecuado)
  Bartlett p   : 0.0000  (significativo ✓)
  ✗ Error en scree: check_array() got an unexpected keyword argument 'force_all_finite'. Omitido.

═══════════════════════════════════════════════════════
  BLOQUE: ESCUELA (291 columnas)
═══════════════════════════════════════════════════════
  KMO          : 0.265  (bajo — interpretar con cautela)
  Bartlett p   : 0.0000  (significativo ✓)
  ✗ KMO < 0.5 — bloque no apto para EFA. Omitido.

═══════════════════════════════════════════════════════
  BLOQUE: PADRES (1 columnas)
═══════════════════════════════════════════════════════
  ✗ Bloque con menos de 5 columnas — omitido.

═══════════════════════════════════════════════════════
  BLOQUE: OTROS (138 columnas)
═══════════════════════════════════════════════════════
  KMO          : 0.841  (adecuado)
  Bartlett 

## 6. Resumen del EFA

In [10]:
print("═" * 60)
print(f"  {'Bloque':<18} {'Cols':>6} {'Factores':>10} {'Seleccionados':>14}")
print("═" * 60)
for bloque, res in resultados_efa.items():
    print(f"  {bloque:<18} {res['n_columnas_bloque']:>6} "
          f"{res['n_factores']:>10} {len(res['items_seleccionados']):>14}")
print("─" * 60)
print(f"  {'TOTAL ítems seleccionados':<40} {len(todos_items_seleccionados):>14}")
print("═" * 60)

════════════════════════════════════════════════════════════
  Bloque               Cols   Factores  Seleccionados
════════════════════════════════════════════════════════════
  estudiante            162          0              0
  escuela               291          0              0
  padres                  1          0              0
  otros                 138          0              0
────────────────────────────────────────────────────────────
  TOTAL ítems seleccionados                             0
════════════════════════════════════════════════════════════


## 7. Exportar cargas factoriales completas

Útil para la sección de metodología del artículo.

In [11]:
frames_cargas = []
for bloque, res in resultados_efa.items():
    if not res["cargas"].empty:
        df_c = res["cargas"].copy()
        df_c["bloque"] = bloque
        df_c["carga_max"] = df_c.drop(columns="bloque").abs().max(axis=1)
        df_c["seleccionado"] = df_c["carga_max"] >= UMBRAL_CARGA
        frames_cargas.append(df_c)

if frames_cargas:
    df_cargas_global = pd.concat(frames_cargas)
    df_cargas_global.to_csv("efa_cargas_factoriales.csv")
    print("Guardado: efa_cargas_factoriales.csv")

## 8. Guardar dataset final para IRT y LCA

In [ ]:
# Eliminar duplicados por si un ítem apareció en más de un bloque
items_unicos = list(dict.fromkeys(todos_items_seleccionados))

df_final = df[items_unicos].copy()
df_final.to_csv("dataset_efa_items_seleccionados.csv", index=False)

print(f"Guardado: dataset_efa_items_seleccionados.csv")
print(f"  Filas : {df_final.shape[0]:,}")
print(f"  Ítems : {df_final.shape[1]:,}")
print()
print("Siguiente paso → paso3_irt.ipynb  y  paso3_lca.ipynb")

## 9. ── OPCIONAL ── Ajuste manual del número de factores por bloque

Si el scree plot sugiere un número diferente al automático, puedes forzarlo aquí y re-ejecutar solo ese bloque.

In [ ]:
# Ejemplo: forzar 3 factores en el bloque 'estudiante'
# Descomenta y ajusta según lo que veas en los scree plots

# AJUSTES_MANUALES = {
#     "estudiante" : 3,
#     "escuela"    : 2,
# }
#
# for bloque, n_fact_manual in AJUSTES_MANUALES.items():
#     cols_bloque = [c for c, b in bloque_por_col.items() if b == bloque]
#     df_bloque = df[cols_bloque].copy()
#     items, cargas, n_fact = efa_bloque(
#         df_bloque, nombre_bloque=bloque,
#         umbral_carga=UMBRAL_CARGA,
#         max_factores=n_fact_manual
#     )
#     resultados_efa[bloque]["items_seleccionados"] = items
#     resultados_efa[bloque]["cargas"]              = cargas
#     resultados_efa[bloque]["n_factores"]          = n_fact